In [2]:
import numpy as np

def transaction_costs(trade_values, etf_spreads, portfolio_value):
    """
    trade_values: array of € amounts traded per ETF
    etf_spreads: half-spread % per ETF
    """
    # 1. Brokerage (IBKR tiered: 0.05% min €3.50)
    brokerage = np.maximum(3.50, 0.0005 * trade_values)
    
    # 2. Exchange + clearing (~€1.90 per trade)
    exchange = 1.90 * (trade_values > 0)
    
    # 3. Spread costs (round-trip = 2 * half-spread)
    spreads = 2 * etf_spreads * trade_values
    
    total_costs = brokerage + exchange + spreads
    return np.sum(total_costs) / portfolio_value  # % drag

# ETFs spreads (bps) - estimated
etf_spreads = np.array([0.0075, 0.050, 0.100, 0.035, 0.130, 0.030, 0.030]) / 100

# # In backtest:
# prev_weights = np.ones(8) / 8  # initial equal weight
# total_tc_costs = []

# for w_opt in weights_opt:
#     tc_cost, turnover = transaction_costs(prev_weights, w_opt, 100000, etf_spreads)
#     total_tc_costs.append(tc_cost)
#     returns_with_costs = returns - tc_cost  # subtract from period return
#     prev_weights = w_opt

def realistic_rebalance(portfolio_value, target_weights, prev_weights, etf_prices, prev_shares, weight_diff_rebalance):
    """
    prev_shares: (N,) current shares held (persistent across calls)
    Returns: new_shares, trade_values, tc_drag, actual_weights
    """

    diff = np.abs(prev_weights - target_weights)
    
    # Target values excluding cash
    target_etf_values = portfolio_value * target_weights[:-1]
    
    # Target shares (not rounded)
    target_shares = target_etf_values / etf_prices
    
    # Trades required
    shares_traded = np.abs(target_shares - prev_shares)
    trade_values = shares_traded * etf_prices

    if diff[diff >= weight_diff_rebalance].any():
        return target_shares, trade_values, 0.0, target_weights, portfolio_value
    
    # Transaction cost drag
    tc_drag = transaction_costs(trade_values, etf_spreads, portfolio_value)
    
    # Actual post-trade portfolio value (after costs)
    new_portfolio_value = portfolio_value - tc_drag * portfolio_value
    
    # Actual weights (including rounding error)
    actual_etf_values = target_shares * etf_prices
    cash_value = new_portfolio_value - np.sum(actual_etf_values)
    actual_weights = np.append(actual_etf_values / new_portfolio_value, cash_value / new_portfolio_value)
    actual_weights = np.array(actual_weights)
    
    return target_shares, trade_values, tc_drag, actual_weights, new_portfolio_value

In [3]:
import pandas as pd
from pathlib import Path
import plotly.graph_objects as go

snp = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "CSPX ETF Stock Price History.csv")
china = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "CNYA ETF Stock Price History.csv")
em = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "EIMI ETF Stock Price History.csv")
gold = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "XAD5 ETF Stock Price History.csv")
india = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "INR ETF Stock Price History.csv")
mscieurope = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "XMEU ETF Stock Price History.csv")
smallcapeurope = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "SXRJ ETF Stock Price History.csv")

dfs = [snp, china, em, gold, india, mscieurope, smallcapeurope]
date_sets = [set(df["Date"]) for df in dfs]

# Find the intersection of all date sets
common_dates = set.intersection(*date_sets)

# Filter each DataFrame to include only common dates
dfs = [df[df["Date"].isin(common_dates)] for df in dfs]

N = len(dfs)
T = len(dfs[0]) - 1

def correct_volume(vol_value):
    if isinstance(vol_value, (float, int)):
        return vol_value
    if vol_value[-1] == "K":
        return 1000 * float(vol_value[:-1])
    if vol_value[-1] == "M":
        return 1_000_000 * float(vol_value[:-1])
    return float(vol_value)

returns_all = np.empty((T, N))
sorted_returns_all = np.empty((T, N))
volumes_all = np.empty((T, N))
prices_all = np.empty((T, N))
dfs_new = []
for i, df in enumerate(dfs):
    df = df[::-1]
    df["Price"] = pd.to_numeric(df["Price"], errors="coerce")  # Handles str/NaN/object
    returns = df["Price"].pct_change().values[1:]
    returns_all[:, i] = returns
    sorted_returns_all[:, i] = sorted(returns)
    volumes_all[:, i] = df["Vol."].apply(correct_volume).values[1:]
    prices_all[:,i] = df["Price"].values[1:]
    dfs_new.append(df)

all_dates = dfs[0]["Date"].values[1:]
volumes_all = np.nan_to_num(volumes_all, nan=0.0)

In [4]:
import tensorflow as tf

def max_drawdown(prices, weights):
    # prices (T,N)
    # weights (N,)
    T, N = prices.shape
    portfolio_values = prices @ weights # (T,)
    highest_peak = -1
    worst_pct = 0.0
    for t in range(T):
        highest_peak = max(highest_peak, portfolio_values[t])
        worst_pct = min(worst_pct, portfolio_values[t] / highest_peak - 1)
    return worst_pct

def perform_validation(w, validation_data, validation_prices):
    # validation data is shape (T, N)

    cash_returns = np.zeros((validation_data.shape[0], 1))  # or use risk-free rate
    validation_data_with_cash = np.hstack([validation_data, cash_returns])
    #print('shape of validation_data_with_cash:', validation_data_with_cash.shape)
    #print('weights shape:', w.shape)

    # Reshape w to (8, 1) for matrix multiplication
    w = tf.reshape(w, (-1, 1))

    # perform metrics
    total_return = validation_data_with_cash @ w  # shape (T, 1)
    total_return = total_return.numpy().flatten()  # convert to (T,)

    cumulative_returns = np.cumprod(total_return + 1) - 1
    risk_free = 0.0
    std = np.std(total_return)
    downside_std = np.std(total_return[total_return < 0])
    sharpe = (255 * np.mean(total_return) - risk_free) / (np.sqrt(255) * std)
    sortino = (255 * np.mean(total_return) - risk_free) / (np.sqrt(255) * downside_std)

    alpha = 0.10

    var_alpha = np.quantile(total_return, alpha)
    cvar = total_return[total_return <= var_alpha].mean()
    mean_return = np.mean(total_return)

    md = max_drawdown(validation_prices, w[:-1].numpy())

    return var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return, md
    

C:\Users\tobia\AppData\Roaming\Python\Python311\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\tobia\AppData\Roaming\Python\Python311\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\tobia\AppData\Roaming\Python\Python311\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/resource_handle.proto. Please 

In [5]:
def lower_part_mean(matrix):
    n, _ = matrix.shape
    s = 0.0
    for i in range(1,n):
        for j in range(i):
            s += matrix[i,j]
    N = (n-1)*n/2
    return s / N

def compute_statistics_rolling(returns, window_size, stepsize):
    print('returns:'); print(returns)
    all_corrs = []
    all_volas = []
    T, N = returns.shape
    w = np.ones(N) / N
    #print('T:', T)
    #print('testetst')
    for idx in range(0, T - window_size, stepsize):
        section = returns[idx:idx+window_size,:]
        #print('section shape:', section.shape)
        corr = np.corrcoef(section, rowvar=False) # (N,N)
        #print('appending', lower_part_mean(corr))
        all_corrs.append(lower_part_mean(corr))
        returns_total = section @ w
        #print('returns_total:', returns_total)
        std = np.std(returns_total)
        all_volas.append(std)
    
    all_corrs = np.array(all_corrs)
    all_volas = np.array(all_volas)
    #print('result:'); print(all_corrs)
    return all_corrs, all_volas


In [6]:
# run

corrs, volas = compute_statistics_rolling(returns_all, 90, 30)

print('mean corrs:', np.mean(corrs))
print('mean volas:', np.mean(volas))
print('highest corrs:', np.max(corrs))
print('lowest corrs:', np.min(corrs))
print('lowest volas:', np.min(volas))
print('highest volas:', np.max(volas))
# results:
# based on this, determine threshold for correlation and volatility

vola_thresh = np.percentile(volas, 75)
corr_thresh = np.percentile(corrs, 75)

corrs_mean = np.mean(corrs)
corrs_std = np.std(corrs)
volas_mean = np.mean(volas)
volas_std = np.std(volas)

returns:
[[ 0.00583242 -0.00188324 -0.00078462 ...  0.00178147  0.00061637
   0.00415324]
 [ 0.00041419  0.03962264  0.01177856 ... -0.02489627 -0.00517433
  -0.01030928]
 [-0.0100916  -0.03085299 -0.01590997 ... -0.01823708 -0.0152322
  -0.01896208]
 ...
 [-0.00759045 -0.01013514 -0.00854345 ... -0.01162325 -0.00697306
   0.00134771]
 [-0.0127289  -0.01194539 -0.01723413 ... -0.01662612 -0.02734333
  -0.02706744]
 [-0.00528727 -0.01208981 -0.01796407 ... -0.01773196 -0.01154014
  -0.01629265]]
mean corrs: 0.3823000782738932
mean volas: 0.008017332792505716
highest corrs: 0.6354282873413744
lowest corrs: 0.18268267958087434
lowest volas: 0.003908344756921926
highest volas: 0.021881164993669322


In [7]:
import tensorflow as tf

learning_rate = 0.01
alpha = 0.05   # tail probability for CVaR
num_steps = 2000
min_weight = 1/9 # 1/N = 1/7, so a little less
max_weight = 0.3
a = 1.0 # for stress environment tuning
b = 1.0 # for stress environment tuning
weight_diff_rebalance = 0.10 # minimum weight difference for a portfolio rebalance
portfolio_value = 10_000

def sigmoid(x):
    return 1/(1+np.exp(-x))

def normalize(x):
    return (x - tf.reduce_mean(x)) / (tf.math.reduce_std(x) + 1e-8)

def rank_transform(x):
    ranks = tf.argsort(tf.argsort(x))
    return tf.cast(ranks, tf.float32) / tf.cast(tf.size(x), tf.float32)

def gradient(train_data, vol_data, params):

    lambda_, gamma, tau = params

    # returns (weights, mode, had_to_cap_weights, stress_factor, turnover_values)
    # where mode = 0 if returned normal (optimal) weights,
    # 1 = if returned equal because of correlations
    # 2 = if returned equal because of volatility

    # train data: (T,N)
    w = tf.Variable(tf.ones([N], dtype=tf.float32) / N)
    t = tf.Variable(0.0, dtype=tf.float32)  # VaR threshold

    optimizer = tf.keras.optimizers.Adam(learning_rate)

    turnover_values = []

    X_tf = tf.convert_to_tensor(train_data, dtype=tf.float32) # (T,N)
    V_tf = tf.convert_to_tensor(vol_data, dtype=tf.float32) # (T,N)
    #cov_tf = tf.convert_to_tensor(cov, dtype=tf.float32)  # shape (N, N)

    corr = np.corrcoef(train_data, rowvar=False)
    mean_corr = lower_part_mean(corr)
    #print('mean corr:', mean_corr)
    corr_standardized = (mean_corr - corrs_mean) / corrs_std

    w_prev = tf.identity(w)

    for step in range(num_steps):
        with tf.GradientTape() as tape:
            # enforce sum-to-1 via softmax
            w_normalized = tf.nn.softmax(w)
            w_col = tf.reshape(w_normalized, [N, 1])  # shape (N,1)

            # portfolio returns
            portfolio_returns = tf.matmul(X_tf, tf.reshape(w_col, [-1,1])) # (T)
            losses = -portfolio_returns
            cvar = t + (1 / (1 - alpha)) * tf.reduce_mean(tf.nn.relu(losses - t))

            # momentum per asset (N,)
            momentum = tf.reduce_mean(X_tf[-60:], axis=0)

            # volume per asset (N,)
            vol_short = tf.reduce_mean(V_tf[-10:], axis=0)
            vol_long  = tf.reduce_mean(V_tf[-60:], axis=0)
            volume_signal = (vol_short - vol_long) / (vol_long + 1e-8)

            volatility = tf.math.reduce_std(X_tf[-60:], axis=0)
            risk_adj_momentum = momentum / (volatility + 1e-8)

            m1 = normalize(momentum)
            m2 = normalize(risk_adj_momentum)
            m3 = normalize(volume_signal)
            #print('m1, m2, m3:', m1, m2, m3)

            # combine (N,)
            mu = 0.5 * m1 + 0.3 * m2 + 0.2 * m3
            mu = rank_transform(mu)

            turnover_penalty = tau * tf.reduce_sum(tf.square(w - w_prev))
            turnover = tf.reduce_sum(
                tf.abs(w_normalized - w_prev)
            ).numpy()
            turnover_values.append(turnover)

            expected_return = tf.tensordot(w_normalized, mu, axes=1)
            weights_penalty = gamma * tf.reduce_sum(tf.math.square(w_col))

            market_return = tf.reduce_mean(X_tf[-20:])   # avg across assets
            market_vol = tf.math.reduce_std(X_tf[-20:])

            objective = -(expected_return - lambda_ * cvar) + weights_penalty + turnover_penalty

        # compute gradients
        grads = tape.gradient(objective, [w, t])
        optimizer.apply_gradients(zip(grads, [w, t]))

        w_prev = tf.identity(w)

    # after the optimization loop
    optimal_w = tf.nn.softmax(w).numpy()

    # compute volatility
    combined_returns = train_data @ optimal_w # (T,)
    volatility = np.std(combined_returns)
    volatility_standardized = (volatility - volas_mean) / volas_std

    stress_score = sigmoid(a * corr_standardized + b * volatility_standardized)
    chosen_w = stress_score * np.ones(N) / N + (1 - stress_score) * optimal_w

    crash_signal = tf.sigmoid(-5 * market_return + 5 * market_vol)
    market_return = tf.reduce_mean(X_tf[-20:])   # avg across assets
    crash_signal = tf.sigmoid(-5 * market_return + 5 * market_vol)
    cash_weight = crash_signal * 0.5  # up to 50% cash
    w_risky = (1 - cash_weight) * chosen_w
    w_final = tf.concat([w_risky, [cash_weight]], axis=0)

    if tf.reduce_any(w_final <= min_weight):
        w_final = np.clip(w_final, a_min = min_weight, a_max = max_weight)
        return w_final / np.sum(w_final), True, stress_score, turnover_values
    return w_final.numpy(), False, stress_score, turnover_values # changed from mean to actual turnover

In [10]:
# run

# λ=0.8, γ=0.08, τ=0.75  # Best Sharpe + highest turnover penalty

lambda_ = 0.8
gamma = 0.08
tau = 0.75

num_rows_back = 252 * 4 # last 4 years
window_returns = returns_all[returns_all.shape[0]-num_rows_back:]
window_volume = volumes_all[returns_all.shape[0]-num_rows_back:]

# use this weights set
w = [0.1137606, 0.09441126, 0.09441126, 0.14128062, 0.09441126, 0.11370315,
0.12906162, 0.21896023]
# in this order: 
# dfs = [snp, china, em, gold, india, mscieurope, smallcapeurope, cash]

( # replaced w with actual_weights
    var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return, md
) = perform_validation(w, window_returns, window_volume) # same data as train (training results)

print('sharpe:', sharpe)
print('sortino:', sortino)
print('cvar:', cvar)
print('var:', var_alpha)

sharpe: 0.996010714153967
sortino: 1.351530938556092
cvar: -0.009710199
var: -0.005979722


In [11]:
# compute volatility
w = np.array(w)
returns_sequence = window_returns @ w[:-1]
volatility = np.std(returns_sequence)
print('volatility:', volatility)

# best idea imo: put cash into bonds

volatility: 0.005353691933350477


In [ ]:
def full_evaluation(returns, volumes, start_weights, rebalancing_period, lambda_, gamma, tau):
    T, N = returns.shape
    final_returns = np.empty(T)
    for i in range(0, T, rebalancing_period):
        if i == 0:
            weights = start_weights
        else:
            train_returns = returns[i-rebalancing_period:i]
            train_volume = volumes[i:min(T, i+rebalancing_period)]
            weights, capped_weights, stress, turnover_list = gradient(train_returns, train_volume, [lambda_, gamma, tau])

        validation_returns = returns[i:min(T, i+rebalancing_period)]
        weights_without_cash = weights[:-1]
        #var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return, md = perform_validation(weights, validation_returns, validation_prices)
        final_returns[i:min(T, i+rebalancing_period)] = validation_returns @ weights_without_cash

    return final_returns


In [18]:
rebalance_period = 90 # about 3 months
returns = full_evaluation(returns_all, volumes_all, w, rebalance_period, lambda_, gamma, tau)

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(len(returns))), y=returns))
fig.show()

shapes: (90, 7) (8,)
shapes: (90, 7) (8,)
shapes: (90, 7) (8,)
shapes: (90, 7) (8,)
shapes: (90, 7) (8,)
shapes: (90, 7) (8,)
shapes: (90, 7) (8,)
shapes: (90, 7) (8,)
shapes: (90, 7) (8,)
shapes: (90, 7) (8,)
shapes: (90, 7) (8,)
shapes: (90, 7) (8,)
shapes: (90, 7) (8,)
shapes: (90, 7) (8,)
shapes: (90, 7) (8,)
shapes: (90, 7) (8,)
shapes: (90, 7) (8,)
shapes: (90, 7) (8,)
shapes: (90, 7) (8,)
shapes: (90, 7) (8,)
shapes: (90, 7) (8,)
shapes: (90, 7) (8,)
shapes: (90, 7) (8,)
shapes: (90, 7) (8,)
shapes: (90, 7) (8,)
shapes: (90, 7) (8,)
shapes: (90, 7) (8,)
shapes: (90, 7) (8,)
shapes: (6, 7) (8,)
